# Generate Gene Transcription Dataset for Replicate 1 and Replicate 2 of Yulong's 2017 Cell Cycle Experiment

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import matplotlib.pyplot as plt

import numpy as np
import pandas as pd


In [9]:
import os

# First make a dataframe that contains the metadata and filepaths
# for the RNA-seq data. This will make things convenient
# for when we want to read from disk
path = '/Users/trung/Research/_archive/data/bam/cell_cycle/rna/'
rows = []
for filename in os.listdir(path):
    if filename.endswith('bam'):
        fil_spl = filename.split('_')
        row = {'replicate': fil_spl[2].replace('rep', ''), 
               'time': fil_spl[3], 'full_path': path + filename}
        rows.append(row)

bam_df = pd.DataFrame.from_records(rows)
bam_df['time'] = bam_df['time'].astype(int)
bam_df['replicate'] = bam_df['replicate'].astype(int)
bam_df = bam_df.sort_values(['replicate', 'time'])
bam_df

,replicate,time,full_path
14,1,0,/Users/trung/Research/_archive/data/bam/cell_c...
12,1,20,/Users/trung/Research/_archive/data/bam/cell_c...
13,1,30,/Users/trung/Research/_archive/data/bam/cell_c...
1,1,40,/Users/trung/Research/_archive/data/bam/cell_c...
3,1,50,/Users/trung/Research/_archive/data/bam/cell_c...
17,1,60,/Users/trung/Research/_archive/data/bam/cell_c...
25,1,70,/Users/trung/Research/_archive/data/bam/cell_c...
0,1,80,/Users/trung/Research/_archive/data/bam/cell_c...
9,1,90,/Users/trung/Research/_archive/data/bam/cell_c...
5,1,100,/Users/trung/Research/_archive/data/bam/cell_c...


In [3]:
from cc_src.sgd import read_nondubious_genes_dataset

# Here is the list of genes we will be using for all of our deconvolutions
orfs = read_nondubious_genes_dataset()
orfs.head(2)

,gene,chr,cat,start,stop,strand,classification,length,TSS,PAS,manually_curated,promoter_start,promoter_end,gene_body_start,gene_body_end
orf_name,,,,,,,,,,,,,,,
YAL068C,PAU8,1,gene,1807,2169,-,Verified,362,2169,NaN,NaN,2169.0,2469.0,1669.0,2169.0
YAL067W-A,YAL067W-A,1,gene,2480,2707,+,Uncharacterized,227,2480,NaN,NaN,2180.0,2480.0,2480.0,2980.0


In [4]:
from src.timer import Timer
from cc_src.read_bam import read_rna_bam
from cc_src.transcription import calculate_read_counts
from cc_src.transcription import convert_to_TPM_all_times
    
def get_read_counts_TPM(replicate_bam):

    timer = Timer()
    all_times_read_counts = orfs[[]].copy()

    for _, row in replicate_bam.iterrows():

        time = row.time
        print(f"Reading BAM file for time {time} minutes")

        rna_reads = read_rna_bam(row.full_path, time, timer, log=True)
        orf_reads = calculate_read_counts(orfs, rna_reads)
        all_times_read_counts.loc[:, time] = orf_reads

        print(f"Done. {timer.get_time()}")
    
    TPMs = convert_to_TPM_all_times(all_times_read_counts, orfs['length'])

    return all_times_read_counts, TPMs

In [5]:
replicate_bam = bam_df[bam_df.replicate == 1]
rep1_read_counts, rep1_TPMs = get_read_counts_TPM(replicate_bam)


Reading BAM file for time 0 minutes
1..Done. 00:00:04.55


In [7]:
rep1_TPMs

,0
orf_name,
YAL068C,6.360031
YAL067W-A,0.000000
YAL067C,339.984890
YAL065C,29.822942
YAL064W-B,145.410388
...,...
YPR200C,0.000000
YPR201W,0.000000
YPR202W,0.000000


In [8]:
replicate_bam = bam_df[bam_df.replicate == 2]
rep2_read_counts, rep2_TPMs = get_read_counts_TPM(replicate_bam)

rep2_TPMs

Reading BAM file for time 0 minutes
1..Done. 00:00:04.09


,0
orf_name,
YAL068C,8.829055
YAL067W-A,0.000000
YAL067C,558.109292
YAL065C,57.960686
YAL064W-B,142.984219
...,...
YPR200C,0.000000
YPR201W,0.000000
YPR202W,0.000000
